In [ ]:
import wandb
wandb.login()

wandb: Currently logged in as: aliu917 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Colab

In [ ]:
import os
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
DRIVE_PATH = '/content/gdrive/MyDrive/cs329h/code'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/cs329h'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

In [ ]:
%cd cs329h

/content/gdrive/MyDrive/cs329h/code


In [ ]:
import sys
sys.path.append("/content/gdrive/MyDrive/cs329h/code")

In [ ]:
!pip install -U bitsandbytes
%pip install -r requirements_colab.txt

In [ ]:
import argparse
import sys

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
import wandb
import os
import numpy as np

from custom_dpo import dpo_step
from dpo_dataset import DpoJsonlDataset
from tqdm import tqdm

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
def dpo_collate(batch, tokenizer, add_generation_prompt=False):
    chosen_texts = [
        [{"role": "user", "content": item["prompt"]},
         {"role": "assistant", "content": item["chosen_text"]}]
        for item in batch
    ]
    rejected_texts = [
        [{"role": "user", "content": item["prompt"]},
         {"role": "assistant", "content": item["rejected_text"]}]
        for item in batch
    ]
    sample_fracs = [item["sample_frac"] for item in batch]

    chosen_encoding_ids = tokenizer.apply_chat_template(
        chosen_texts, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    rejected_encoding_ids = tokenizer.apply_chat_template(
        rejected_texts, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    chosen_encodings = [{"input_ids": ids} for ids in chosen_encoding_ids]
    rejected_encodings = [{"input_ids": ids} for ids in rejected_encoding_ids]

    chosen_batch = tokenizer.pad(chosen_encodings, padding=True, return_tensors="pt")
    rejected_batch = tokenizer.pad(rejected_encodings, padding=True, return_tensors="pt")
    # metadata = [item["metadata"] for item in batch]

    return {
        "chosen": chosen_batch,
        "rejected": rejected_batch,
        "sample_frac": torch.tensor(sample_fracs),
    }

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for potentially higher accuracy (optional)
    bnb_4bit_quant_type="nf4",  # Quantization type (specifics depend on hardware and library)
    bnb_4bit_compute_dtype=torch.bfloat16  # Compute dtype for improved efficiency (optional)
)

# RUN CONFIG

In [ ]:
run_name = "run_toy_life_5e5_0p01_sampled_ips_newwords_50"

In [ ]:
default_config = {
    "model_name": "google/gemma-2-2b-it",
    "device": device,
    "learning_rate": 5e-5,
    "num_epochs": 50,
    "beta": 0.01,
    "max_seq_len": 100,
    "batch_size": 1,
    "ips": True,
    "sample_data": True,
}

In [ ]:
seed = 1

np.random.seed(seed)
torch.manual_seed(seed)

if device == "cuda":
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
run = wandb.init(
    project="dpo-finetune",
    name=run_name,
    config=default_config
)
if wandb.config.sample_data:
    path = "data/sampled_dpo_toy_life2.jsonl"
else:
    path = "data/all_dpo_toy_life2.jsonl"


In [ ]:
model_name = default_config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ.get("HUGGINGFACE_TOKEN"))
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get("HUGGINGFACE_TOKEN"),
    # low_cpu_mem_usage=True
    quantization_config=bnb_config,
)
ref_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

# Training

In [ ]:
train_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get("HUGGINGFACE_TOKEN"),
    # low_cpu_mem_usage=True,
    quantization_config=bnb_config,
)
lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
train_model = get_peft_model(train_model, lora_config)
train_model.train()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
           

In [ ]:
def sample_model(tokenizer, model, prompt, N=100):
    """
    Samples N different completions from the model based on the given prompt.

    Args:
    tokenizer: The tokenizer object used to encode/decode text.
    model: The language model used for generation.
    prompt (str): The input prompt for which completions will be generated.
    N (int): The number of completions to generate.

    Returns:
    list[str]: A list of N generated completions.
    """

    chat = [{"role": "user", "content": prompt}]
    chat_tokens = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

    # Generate N different responses
    outputs = model.generate(
        torch.tensor([chat_tokens], device=model.device),
        num_return_sequences=N,
        max_new_tokens=32,
        temperature=0.15,
        top_k=50,
        top_p=0.95,
        do_sample=True
    )

    def extract_response(decoded_text):
        return decoded_text.rsplit('model\n', 1)[-1][:-2]

    responses = [extract_response(tokenizer.decode(output, skip_special_tokens=True)) for output in outputs]
    return responses

In [ ]:
from collections import Counter

def fraction_responses(responses):
  response_words = [r.strip().split(" ") for r in responses]
  # 0) the 1) sky 2) appears 3) blue

  one_word_distr = [words[0].strip().strip("*") for words in response_words if len(words) > 0]
  one_word_counter = Counter(one_word_distr)
  distribution = {word: count / 100 for word, count in one_word_counter.items()}
  print(distribution)

  # two_word_distr = [words[4] + " " + words[5] for words in response_words if len(words) > 5]
  # two_word_counter = Counter(two_word_distr)
  # distribution = {word: count / 100 for word, count in two_word_counter.items()}
  # print(distribution)

In [ ]:
prompt = "Describe life in one word."

prior_responses = sample_model(tokenizer, train_model, prompt)
print('Sampled responses before fine-tuning:\n' + '\n'.join(prior_responses[:100]))
print(f'Fraction responses with because of: {fraction_responses(prior_responses)}')

Sampled responses before fine-tuning:
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Evolving**
**Dynamic**
**Evolving**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dynamic**
**Dyna

In [ ]:
dataset = DpoJsonlDataset(path, tokenizer=tokenizer, max_length=wandb.config.max_seq_len)
dataloader = DataLoader(dataset, batch_size=wandb.config.batch_size, shuffle=True, num_workers=0, collate_fn=lambda batch: dpo_collate(batch, tokenizer))
optimizer = torch.optim.Adam(train_model.parameters(), lr=wandb.config.learning_rate)

In [ ]:
list(dataset)

[{'prompt': 'Describe life in one word.',
  'chosen_text': 'Change',
  'rejected_text': 'Journey',
  'sample_frac': 0.05},
 {'prompt': 'Describe life in one word.',
  'chosen_text': 'Journey',
  'rejected_text': 'Dynamic',
  'sample_frac': 1.0},
 {'prompt': 'Describe life in one word.',
  'chosen_text': 'Journey',
  'rejected_text': 'Dynamic',
  'sample_frac': 1.0},
 {'prompt': 'Describe life in one word.',
  'chosen_text': 'Journey',
  'rejected_text': 'Evolving',
  'sample_frac': 1.0},
 {'prompt': 'Describe life in one word.',
  'chosen_text': 'Journey',
  'rejected_text': 'Flow',
  'sample_frac': 1.0}]

In [ ]:
from custom_dpo import get_logprobs
import torch.nn.functional as F

def compute_dpo_objective(preferred_train_logprobs, nonpreferred_train_logprobs, preferred_ref_logprobs, nonpreferred_ref_logprobs, beta, ips_weight=None):
    """
    Computes the Direct Preference Optimization (DPO) objective for training.

    Args:
    preferred_train_logprobs (torch.Tensor): Token probabilities for the preferred chat sequence from the training model.
    nonpreferred_train_logprobs (torch.Tensor): Token probabilities for the non-preferred chat sequence from the training model.
    preferred_ref_logprobs (torch.Tensor): Token probabilities for the preferred chat sequence from the reference model.
    nonpreferred_ref_logprobs (torch.Tensor): Token probabilities for the non-preferred chat sequence from the reference model.
    beta (float): Controls the KL strength of staying close to the reference model.

    Returns:
    torch.Tensor: The computed DPO objective, which is a float.
    """

    # YOUR CODE HERE (~4-6 lines)
    preferred_log_ratio = torch.sum(preferred_train_logprobs, dim=1) - torch.sum(preferred_ref_logprobs, dim=1)
    nonpreferred_log_ratio = torch.sum(nonpreferred_train_logprobs, dim=1) - torch.sum(nonpreferred_ref_logprobs, dim=1)
    dpo_obj = -F.logsigmoid(beta * preferred_log_ratio - beta * nonpreferred_log_ratio)
    if ips_weight:
        dpo_obj_ips = (dpo_obj * ips_weight).mean()
    else:
        dpo_obj_ips = dpo_obj.mean()
    # END OF YOUR CODE

    return dpo_obj_ips


def dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, beta, ips_weight):
    """
    Fine-tunes the training model using DPO. Make sure to disable gradients on the reference model!

    Args:
    optimizer: The optimizer for updating the training model's parameters.
    train_model: The model being fine-tuned.
    ref_model: The reference model.
    preferred_chat_ids (list[int]): The token IDs representing the preferred chat sequence.
    nonpreferred_chat_ids (list[int]): The token IDs representing the non-preferred chat sequence.
    num_gradient_steps (int): The number of gradient updates to perform.
    beta (float): A parameter used in computing the DPO objective.

    Returns:
    dpo loss
    """
    preferred_train_logprobs = get_logprobs(train_model, preferred_chat_ids, preferred_mask)
    nonpreferred_train_logprobs = get_logprobs(train_model, nonpreferred_chat_ids, nonpreferred_mask)

    # Gradients are not needed for the reference model since we will not be optimizing with respect to it
    with torch.no_grad():
        preferred_ref_logprobs = get_logprobs(ref_model, preferred_chat_ids, preferred_mask)
        nonpreferred_ref_logprobs = get_logprobs(ref_model, nonpreferred_chat_ids, nonpreferred_mask)

    dpo_obj = compute_dpo_objective(preferred_train_logprobs, nonpreferred_train_logprobs, preferred_ref_logprobs, nonpreferred_ref_logprobs, beta, ips_weight)
    return dpo_obj


In [ ]:
train_model.train()
ref_model.eval()

for epoch in range(wandb.config.num_epochs):

    total_loss = 0
    total_weight = 0
    for step, data in enumerate(tqdm(dataloader, total=len(dataloader), desc=f"Epoch {epoch+1}")):
        preferred_chat_ids = data["chosen"]["input_ids"].to(train_model.device)
        preferred_mask = data["chosen"]["attention_mask"].to(train_model.device)
        nonpreferred_chat_ids = data["rejected"]["input_ids"].to(train_model.device)
        nonpreferred_mask = data["rejected"]["attention_mask"].to(train_model.device)

        ips_weight = None
        if wandb.config.ips:
            ips_weight = 1 / data["sample_frac"].to(train_model.device)
            total_weight += ips_weight
        else:
            total_weight += 1

        loss = dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, ips_weight)
        # if ips_weight > 1:
        #     print("ips weight:", ips_weight)
        #     print(loss)
        #     with torch.no_grad():
        #       print(dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, None))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    wandb.log({
        "train_loss": total_loss / total_weight,
        "epoch": epoch+1
    })

    if epoch % 10 == 0:
        with torch.no_grad():
            prior_responses = sample_model(tokenizer, train_model, prompt)
            print(f'Fraction responses with because of: {fraction_responses(prior_responses)}')


Epoch 1: 100%|██████████| 5/5 [00:03<00:00,  1.34it/s]


{'Dynamic': 0.67, 'Evolving': 0.25, 'Flow': 0.08}
Fraction responses with because of: None


Epoch 11: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]


{'Flow': 0.73, 'Dynamic': 0.23, 'Evolving': 0.04}
Fraction responses with because of: None


Epoch 21: 100%|██████████| 5/5 [00:03<00:00,  1.41it/s]


...

Epoch 31: 100%|██████████| 5/5 [00:03<00:00,  1.27it/s]


{'Evolving': 0.02, 'Flow': 0.78, 'ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan': 0.2}
Fraction responses with because of: None


Epoch 41: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]


{'Evolving': 0.78, 'ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan': 0.17, 'Flow': 0.05}
Fraction responses with because of: None


Epoch 50: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]


In [ ]:
prompt = "Describe life in one word."

prior_responses = sample_model(tokenizer, train_model, prompt)
print('Sampled responses before fine-tuning:\n' + '\n'.join(prior_responses[:100]))
print(f'Fraction responses with because of: {fraction_responses(prior_responses)}')

Sampled responses before fine-tuning:
ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan
ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan
ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan
ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan
ChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChangeChan
ChangeC